# 01 TF-IDF Logistic Regression Baseline 1 and Ablation A

Upload `06_model_ready.zip` to `/content/`, run all cells, then download `/content/tfidf_logreg_outputs.zip`. Baseline 1 uses class_weight=None; Ablation A uses class_weight='balanced'.

## Dependency Check

In [ ]:
import importlib.util, subprocess, sys
required = {
    "pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn", "matplotlib": "matplotlib",
    "joblib": "joblib", "tqdm": "tqdm", "tabulate": "tabulate"
}
missing = [pip for mod, pip in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependency check complete. Installed:", missing)


## Paths, Data Loading, and Output Folders

In [ ]:
from pathlib import Path
import zipfile, shutil, json, random, datetime, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

TEXT_COL = "model_text"
LABEL_COL = "label_id"
ID_COL = "final_row_id"
SEEDS = [42, 7, 123]
RUN_SEEDS = [42, 7, 123]  # Temporarily set to [42] for faster smoke runs.
THRESHOLD = 0.5

CONTENT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
DATA_ZIP = CONTENT_DIR / "06_model_ready.zip"
if DATA_ZIP.exists():
    print(f"Found dataset ZIP, extracting: {DATA_ZIP}")
    with zipfile.ZipFile(DATA_ZIP, "r") as zf:
        zf.extractall(CONTENT_DIR)
else:
    print("No /content/06_model_ready.zip found. Looking for extracted dataset.")

def find_data_dir():
    candidates = [
        CONTENT_DIR / "06_model_ready",
        CONTENT_DIR / "model_ready",
        CONTENT_DIR / "data" / "06_model_ready",
        CONTENT_DIR / "thesis-modeling" / "data" / "06_model_ready",
        CONTENT_DIR / "ralf revision and GA model" / "data" / "06_model_ready",
        Path.cwd() / "06_model_ready",
        Path.cwd() / "model_ready",
        Path.cwd() / "data" / "06_model_ready",
    ]
    for base in [CONTENT_DIR, Path.cwd()]:
        candidates.extend([p for p in base.rglob("06_model_ready") if p.is_dir()])
    seen = []
    for p in candidates:
        if p not in seen:
            seen.append(p)
    for p in seen:
        if (p / "clean" / "train_clean.csv").exists():
            return p.resolve()
    raise FileNotFoundError("Could not locate 06_model_ready. Upload 06_model_ready.zip to /content and rerun.")

DATA_DIR = find_data_dir()
BASE_DIR = CONTENT_DIR / "baseline_modeling_outputs"
BASE_DIR.mkdir(parents=True, exist_ok=True)
TRAINED_DIR = BASE_DIR / "trained_models"
RESULTS_DIR = BASE_DIR / "results"
REPORTS_DIR = BASE_DIR / "reports"
METRICS_DIR = RESULTS_DIR / "metrics"
PRED_DIR = RESULTS_DIR / "predictions"
FIGURE_DIR = RESULTS_DIR / "figures"
DEGRADATION_DIR = RESULTS_DIR / "degradation_tables"
for d in [TRAINED_DIR, RESULTS_DIR, REPORTS_DIR, METRICS_DIR, PRED_DIR, FIGURE_DIR, DEGRADATION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SPLIT_PATHS = {
    "train_clean": DATA_DIR / "clean" / "train_clean.csv",
    "val_clean": DATA_DIR / "clean" / "val_clean.csv",
    "test_clean": DATA_DIR / "clean" / "test_clean.csv",
    "train_augmented_ablation_b": DATA_DIR / "augmented_training" / "train_augmented_for_ablation_b.csv",
    "test_adv_10": DATA_DIR / "adversarial_test" / "test_adv_10.csv",
    "test_adv_20": DATA_DIR / "adversarial_test" / "test_adv_20.csv",
    "test_adv_30": DATA_DIR / "adversarial_test" / "test_adv_30.csv",
}
EVAL_SPLITS = ["test_clean", "test_adv_10", "test_adv_20", "test_adv_30"]
for name, path in SPLIT_PATHS.items():
    if name != "train_augmented_ablation_b" and not path.exists():
        raise FileNotFoundError(f"Missing required split {name}: {path}")
print("DATA_DIR =", DATA_DIR)
print("BASE_DIR =", BASE_DIR)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)

def load_split(split_name):
    path = SPLIT_PATHS[split_name]
    if not path.exists():
        raise FileNotFoundError(f"Missing split file: {path}")
    df = pd.read_csv(path)
    assert TEXT_COL in df.columns, f"{path} missing {TEXT_COL}"
    assert LABEL_COL in df.columns, f"{path} missing {LABEL_COL}"
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    if ID_COL not in df.columns:
        df[ID_COL] = np.arange(len(df))
    return df

train_clean_df = load_split("train_clean")
val_clean_df = load_split("val_clean")
test_dfs = {split: load_split(split) for split in EVAL_SPLITS}
print("Loaded rows:", {"train_clean": len(train_clean_df), "val_clean": len(val_clean_df), **{k: len(v) for k, v in test_dfs.items()}})


## Evaluation Helpers

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def binary_metrics(y_true, prob, model_name, split, seed, threshold=THRESHOLD):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob)
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, pred, pos_label=1, average="binary", zero_division=0)
    return {
        "model": model_name, "split": split, "seed": seed, "n_rows": int(len(y_true)),
        "accuracy": accuracy_score(y_true, pred), "precision_smishing": precision,
        "recall_smishing": recall, "f1_smishing": f1,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) else 0.0,
        "false_positive_rate": fp / (fp + tn) if (fp + tn) else 0.0,
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "support_ham": int((y_true == 0).sum()), "support_smishing": int((y_true == 1).sum()),
        "threshold": threshold,
    }

def save_confusion(m, out_path, title):
    mat = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(mat, cmap="Blues")
    ax.set_xticks([0, 1], ["Pred Ham", "Pred Smishing"])
    ax.set_yticks([0, 1], ["True Ham", "True Smishing"])
    ax.set_title(title)
    for i in range(2):
        for j in range(2): ax.text(j, i, str(mat[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout(); fig.savefig(out_path, dpi=160); plt.close(fig)

def degradation_table(metrics_df, model_name):
    def get(split, col):
        row = metrics_df[metrics_df["split"] == split]
        return np.nan if row.empty else float(row.iloc[0][col])
    row = {
        "model": model_name,
        "clean_recall": get("test_clean", "recall_smishing"), "adv10_recall": get("test_adv_10", "recall_smishing"),
        "adv20_recall": get("test_adv_20", "recall_smishing"), "adv30_recall": get("test_adv_30", "recall_smishing"),
        "clean_f1": get("test_clean", "f1_smishing"), "adv10_f1": get("test_adv_10", "f1_smishing"),
        "adv20_f1": get("test_adv_20", "f1_smishing"), "adv30_f1": get("test_adv_30", "f1_smishing"),
        "clean_fnr": get("test_clean", "false_negative_rate"), "adv30_fnr": get("test_adv_30", "false_negative_rate"),
    }
    row["clean_to_adv30_recall_drop"] = row["clean_recall"] - row["adv30_recall"]
    row["clean_to_adv30_f1_drop"] = row["clean_f1"] - row["adv30_f1"]
    row["clean_to_adv30_fnr_increase"] = row["adv30_fnr"] - row["clean_fnr"]
    return pd.DataFrame([row])

def mean_std_metrics(df, model_name):
    metric_cols = ["accuracy", "precision_smishing", "recall_smishing", "f1_smishing", "false_negative_rate", "false_positive_rate"]
    out = df.groupby("split")[metric_cols].agg(["mean", "std"]).reset_index()
    out.columns = ["_".join([x for x in col if x]) for col in out.columns.to_flat_index()]
    out.insert(0, "model", model_name)
    return out


## Train and Evaluate TF-IDF Models

In [ ]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib

TFIDF_MAX_FEATURES_WORD = 5000
TFIDF_MAX_FEATURES_CHAR = 5000

class TextSelector:
    def fit(self, X, y=None): return self
    def transform(self, X): return X

def build_tfidf_pipeline(seed, class_weight):
    features = FeatureUnion([
        ("word_tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), lowercase=True, sublinear_tf=True, min_df=2, max_df=0.95, max_features=TFIDF_MAX_FEATURES_WORD)),
        ("char_tfidf", TfidfVectorizer(analyzer="char", ngram_range=(3, 5), lowercase=True, sublinear_tf=True, min_df=2, max_df=0.95, max_features=TFIDF_MAX_FEATURES_CHAR)),
    ])
    clf = LogisticRegression(class_weight=class_weight, solver="liblinear", max_iter=1000, random_state=seed)
    return Pipeline([("features", features), ("clf", clf)])

def run_tfidf_experiment(model_name, class_weight):
    model_dir = TRAINED_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    all_rows = []
    val_rows = []
    for seed in RUN_SEEDS:
        print(f"Training {model_name} seed={seed} class_weight={class_weight}")
        set_seed(seed)
        pipe = build_tfidf_pipeline(seed, class_weight)
        pipe.fit(train_clean_df[TEXT_COL].tolist(), train_clean_df[LABEL_COL].to_numpy())
        joblib.dump(pipe, model_dir / f"{model_name}_seed{seed}.joblib")
        (model_dir / f"{model_name}_seed{seed}_config.json").write_text(json.dumps({
            "model": model_name, "class_weight": class_weight, "seed": seed,
            "train_split": "train_clean", "validation_split": "val_clean_reporting_only",
            "text_col": TEXT_COL, "label_col": LABEL_COL,
            "word_ngrams": [1, 2], "char_ngrams": [3, 5], "max_features_total": TFIDF_MAX_FEATURES_WORD + TFIDF_MAX_FEATURES_CHAR,
        }, indent=2), encoding="utf-8")
        val_prob = pipe.predict_proba(val_clean_df[TEXT_COL].tolist())[:, 1]
        val_rows.append(binary_metrics(val_clean_df[LABEL_COL], val_prob, model_name, "val_clean", seed))
        for split, df in test_dfs.items():
            prob = pipe.predict_proba(df[TEXT_COL].tolist())[:, 1]
            m = binary_metrics(df[LABEL_COL], prob, model_name, split, seed)
            all_rows.append(m)
            pred = (prob >= THRESHOLD).astype(int)
            pred_df = pd.DataFrame({ID_COL: df[ID_COL], TEXT_COL: df[TEXT_COL], "true_label": df[LABEL_COL], "predicted_label": pred, "predicted_probability": prob})
            pred_df.to_csv(PRED_DIR / f"{model_name}_seed{seed}_predictions_{split}.csv", index=False)
            save_confusion(m, FIGURE_DIR / f"{model_name}_confusion_matrix_seed{seed}_{split}.png", f"{model_name} seed {seed} {split}")
    metrics = pd.DataFrame(all_rows)
    val_metrics = pd.DataFrame(val_rows)
    metrics.to_csv(METRICS_DIR / f"{model_name}_metrics_by_seed.csv", index=False)
    mean_std = mean_std_metrics(metrics, model_name)
    mean_std.to_csv(METRICS_DIR / f"{model_name}_metrics_mean_std.csv", index=False)
    val_metrics.to_csv(METRICS_DIR / f"{model_name}_validation_metrics_by_seed.csv", index=False)
    mean_for_deg = metrics.groupby("split", as_index=False)[["accuracy", "precision_smishing", "recall_smishing", "f1_smishing", "false_negative_rate", "false_positive_rate"]].mean(numeric_only=True)
    degradation_table(mean_for_deg, model_name).to_csv(DEGRADATION_DIR / f"{model_name}_degradation_table.csv", index=False)
    return metrics, mean_std, val_metrics

baseline1_metrics, baseline1_mean_std, baseline1_val = run_tfidf_experiment("tfidf_baseline1", None)
ablation_a_metrics, ablation_a_mean_std, ablation_a_val = run_tfidf_experiment("tfidf_ablation_a", "balanced")


## Reports and Output ZIP

In [ ]:
def write_tfidf_summary(model_name, class_weight, metrics, mean_std):
    deg = pd.read_csv(DEGRADATION_DIR / f"{model_name}_degradation_table.csv")
    report = (
        f"# {model_name} Summary\n\n"
        f"- Training split: train_clean.csv only.\n"
        f"- Validation split: val_clean.csv for reporting only; no threshold tuning.\n"
        f"- Test splits: test_clean, test_adv_10, test_adv_20, test_adv_30 for final evaluation only.\n"
        f"- Logistic Regression class_weight: `{class_weight}`.\n"
        f"- TF-IDF: word n-grams (1,2), char n-grams (3,5), lowercase, sublinear TF, min_df=2, max_df=0.95.\n\n"
        "## Mean/Std Metrics\n\n" + mean_std.to_markdown(index=False) + "\n\n"
        "## Degradation\n\n" + deg.to_markdown(index=False) + "\n"
    )
    (REPORTS_DIR / f"{model_name}_summary.md").write_text(report, encoding="utf-8")

write_tfidf_summary("tfidf_baseline1", None, baseline1_metrics, baseline1_mean_std)
write_tfidf_summary("tfidf_ablation_a", "balanced", ablation_a_metrics, ablation_a_mean_std)

(REPORTS_DIR / "tfidf_logreg_audit_summary.md").write_text(
    "# TF-IDF Logistic Regression Audit Summary\n\n"
    "- Did Baseline 1 use `train_clean.csv` only for training? Yes.\n"
    "- Did Ablation A use `train_clean.csv` only for training? Yes.\n"
    "- Was `class_weight=None` used for Baseline 1? Yes.\n"
    "- Was `class_weight=balanced` used for Ablation A? Yes.\n"
    "- Were test sets used only for final evaluation? Yes. Validation was reporting-only and no test threshold tuning was performed.\n"
    "- How strong is clean-to-adv30 degradation? See degradation tables in `results/degradation_tables/`.\n"
    "- Are there signs the high score may come from strong lexical cues? TF-IDF is lexical by design; high clean scores should be interpreted with adversarial degradation and false-negative counts.\n",
    encoding="utf-8"
)

zip_path = CONTENT_DIR / "tfidf_logreg_outputs.zip"
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for rel in ["trained_models/tfidf_baseline1", "trained_models/tfidf_ablation_a", "results/metrics", "results/predictions", "results/figures", "results/degradation_tables", "reports"]:
        src = BASE_DIR / rel
        if src.exists():
            for file in src.rglob("*"):
                if file.is_file(): zf.write(file, file.relative_to(BASE_DIR))
        else:
            print("Missing for ZIP:", src)
print(f"? TF-IDF output ZIP created: {zip_path}")

required = ["trained_models/tfidf_baseline1", "trained_models/tfidf_ablation_a", "results/metrics", "results/predictions", "results/figures", "results/degradation_tables", "reports"]
with zipfile.ZipFile(zip_path, "r") as zf:
    names = set(zf.namelist())
    for entry in required:
        ok = any(n.startswith(entry.rstrip('/') + '/') for n in names)
        print(f"{entry}: {'FOUND' if ok else 'MISSING'}")
print("Download from Colab sidebar:", zip_path)
